# Minimal Information-Seeking Testbed

## Experiment Overview

This notebook implements the simplest possible epistemic foraging scenario to compare three approaches:

1. **Myopic Reward-Maximizer**: Only considers immediate expected reward
2. **Information Gain ρ-POMDP**: Uses entropy reduction as belief utility
3. **Variational Free Energy ρ-POMDP**: Uses VFE as belief utility

### Environment: Two-State Observation Problem

- **States**: Two hidden states (A or B)
- **Actions**: 
  - `observe`: Pay cost to get noisy observation
  - `commit_A`: Bet that state is A
  - `commit_B`: Bet that state is B
- **Observations**: Noisy signals that partially disambiguate states
- **Rewards**:
  - Correct commitment: +1.0
  - Incorrect commitment: -1.0
  - Each observation: -0.1 (cost)

### Research Question

Do agents using VFE as their utility function exhibit different epistemic foraging behavior compared to information gain or myopic reward maximization?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List, Optional
from dataclasses import dataclass
from enum import Enum
import pandas as pd
from scipy.stats import entropy

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## 1. Environment Implementation

In [ ]:
class Action(Enum):
    """Available actions in the environment."""
    OBSERVE = 0
    COMMIT_A = 1
    COMMIT_B = 2

class State(Enum):
    """Hidden states of the world."""
    A = 0
    B = 1

class Observation(Enum):
    """Observations that hint at the true state."""
    SIGNAL_A = 0
    SIGNAL_B = 1

@dataclass
class EnvConfig:
    """Configuration for the information-seeking environment."""
    observation_accuracy: float = 0.75  # P(correct observation | true state)
    observation_cost: float = 0.1       # Cost per observation
    correct_reward: float = 1.0         # Reward for correct commitment
    incorrect_penalty: float = -1.0     # Penalty for incorrect commitment
    
class MinimalInfoSeekingEnv:
    """Two-state partially observable environment for epistemic foraging.
    
    The agent must decide when to stop observing and commit to a belief
    about which of two states is the true state.
    """
    
    def __init__(self, config: EnvConfig = EnvConfig()):
        self.config = config
        self.true_state = None
        self.done = False
        self.total_reward = 0.0
        self.observation_count = 0
        
    def reset(self) -> None:
        """Reset environment with random true state."""
        self.true_state = np.random.choice([State.A, State.B])
        self.done = False
        self.total_reward = 0.0
        self.observation_count = 0
        
    def step(self, action: Action) -> Tuple[Optional[Observation], float, bool]:
        """Execute action and return (observation, reward, done)."""
        if self.done:
            raise ValueError("Episode already terminated. Call reset().")
        
        if action == Action.OBSERVE:
            obs = self._generate_observation()
            reward = -self.config.observation_cost
            self.observation_count += 1
            self.total_reward += reward
            return obs, reward, False
        
        elif action in [Action.COMMIT_A, Action.COMMIT_B]:
            committed_state = State.A if action == Action.COMMIT_A else State.B
            correct = (committed_state == self.true_state)
            reward = self.config.correct_reward if correct else self.config.incorrect_penalty
            self.total_reward += reward
            self.done = True
            return None, reward, True
        
        else:
            raise ValueError(f"Unknown action: {action}")
    
    def _generate_observation(self) -> Observation:
        """Generate noisy observation based on true state."""
        if np.random.random() < self.config.observation_accuracy:
            # Correct observation
            return Observation.SIGNAL_A if self.true_state == State.A else Observation.SIGNAL_B
        else:
            # Incorrect observation (noise)
            return Observation.SIGNAL_B if self.true_state == State.A else Observation.SIGNAL_A
    
    def get_observation_model(self) -> np.ndarray:
        """Return observation model: P(obs | state).
        
        Returns:
            2x2 matrix where [s, o] = P(observation=o | state=s)
        """
        acc = self.config.observation_accuracy
        return np.array([
            [acc, 1-acc],      # State A: P(SIGNAL_A), P(SIGNAL_B)
            [1-acc, acc]       # State B: P(SIGNAL_A), P(SIGNAL_B)
        ])

## 2. Belief State Tracking

All agents maintain a belief distribution over states using Bayesian updates.

In [ ]:
class BeliefState:
    """Maintains probability distribution over hidden states."""
    
    def __init__(self, initial_belief: Optional[np.ndarray] = None):
        """Initialize belief state.
        
        Args:
            initial_belief: Initial probability distribution [P(A), P(B)]
                          Defaults to uniform [0.5, 0.5]
        """
        self.belief = initial_belief if initial_belief is not None else np.array([0.5, 0.5])
        self.history = [self.belief.copy()]
        
    def update(self, observation: Observation, obs_model: np.ndarray) -> None:
        """Bayesian update given observation.
        
        Args:
            observation: Observed signal
            obs_model: P(obs | state) matrix
        """
        obs_idx = observation.value
        likelihood = obs_model[:, obs_idx]  # P(obs | state) for each state
        
        # Bayes rule: P(state | obs) ∝ P(obs | state) * P(state)
        posterior = likelihood * self.belief
        posterior = posterior / posterior.sum()  # Normalize
        
        self.belief = posterior
        self.history.append(self.belief.copy())
    
    def entropy(self) -> float:
        """Calculate entropy of current belief (uncertainty measure)."""
        return entropy(self.belief, base=2)
    
    def most_likely_state(self) -> State:
        """Return most likely state under current belief."""
        return State.A if self.belief[0] > self.belief[1] else State.B
    
    def confidence(self) -> float:
        """Return confidence in most likely state (max probability)."""
        return np.max(self.belief)
    
    def reset(self) -> None:
        """Reset to uniform belief."""
        self.belief = np.array([0.5, 0.5])
        self.history = [self.belief.copy()]

## 3. Agent Implementations

### 3.1 Base Agent Class

In [ ]:
class BaseAgent:
    """Base class for all agents."""
    
    def __init__(self, env: MinimalInfoSeekingEnv):
        self.env = env
        self.belief = BeliefState()
        self.obs_model = env.get_observation_model()
        
    def reset(self) -> None:
        """Reset agent's belief state."""
        self.belief.reset()
    
    def select_action(self) -> Action:
        """Select action based on agent's policy."""
        raise NotImplementedError
    
    def update_belief(self, observation: Observation) -> None:
        """Update belief given observation."""
        self.belief.update(observation, self.obs_model)
    
    def get_commit_action(self) -> Action:
        """Return commitment action based on current belief."""
        return Action.COMMIT_A if self.belief.most_likely_state() == State.A else Action.COMMIT_B
    
    def expected_reward_of_commit(self) -> float:
        """Calculate expected reward of committing now."""
        confidence = self.belief.confidence()
        return (confidence * self.env.config.correct_reward + 
                (1 - confidence) * self.env.config.incorrect_penalty)

### 3.2 Myopic Reward-Maximizing Agent

Commits when expected reward of committing exceeds cost of observing.

In [ ]:
class MyopicAgent(BaseAgent):
    """Agent that only considers immediate expected reward.
    
    Decision rule:
    - Observe if: E[reward | observe, then commit optimally] > E[reward | commit now]
    - Commit otherwise
    
    This is myopic because it only looks one step ahead.
    """
    
    def select_action(self) -> Action:
        """Select action based on one-step lookahead."""
        # Expected reward of committing now
        commit_value = self.expected_reward_of_commit()
        
        # Expected reward of observing once more, then committing
        observe_value = self._expected_value_of_observe()
        
        if observe_value > commit_value:
            return Action.OBSERVE
        else:
            return self.get_commit_action()
    
    def _expected_value_of_observe(self) -> float:
        """Calculate expected value of observing once, then committing."""
        expected_value = -self.env.config.observation_cost
        
        # For each possible observation
        for obs in [Observation.SIGNAL_A, Observation.SIGNAL_B]:
            # Probability of this observation
            prob_obs = (self.belief.belief * self.obs_model[:, obs.value]).sum()
            
            # Create hypothetical belief after this observation
            temp_belief = BeliefState(self.belief.belief.copy())
            temp_belief.update(obs, self.obs_model)
            
            # Expected reward of committing after this observation
            confidence = temp_belief.confidence()
            commit_reward = (confidence * self.env.config.correct_reward + 
                           (1 - confidence) * self.env.config.incorrect_penalty)
            
            expected_value += prob_obs * commit_reward
        
        return expected_value

### 3.3 Information Gain ρ-POMDP Agent

Uses entropy reduction (information gain) as belief-state utility.

In [ ]:
class InformationGainAgent(BaseAgent):
    """ρ-POMDP agent using information gain as utility.
    
    Utility function:
    ρ(b) = -H(b) = -Σ b(s) log b(s)
    
    Decision rule:
    - Observe if: expected information gain > observation cost
    - Commit otherwise
    """
    
    def __init__(self, env: MinimalInfoSeekingEnv, info_gain_weight: float = 1.0):
        """
        Args:
            env: Environment
            info_gain_weight: Weight on information gain vs reward
        """
        super().__init__(env)
        self.info_gain_weight = info_gain_weight
    
    def select_action(self) -> Action:
        """Select action balancing reward and information gain."""
        commit_value = self.expected_reward_of_commit()
        observe_value = self._expected_value_of_observe_with_info_gain()
        
        if observe_value > commit_value:
            return Action.OBSERVE
        else:
            return self.get_commit_action()
    
    def _expected_value_of_observe_with_info_gain(self) -> float:
        """Expected value including information gain utility."""
        current_entropy = self.belief.entropy()
        expected_value = -self.env.config.observation_cost
        expected_future_entropy = 0.0
        expected_future_reward = 0.0
        
        # For each possible observation
        for obs in [Observation.SIGNAL_A, Observation.SIGNAL_B]:
            prob_obs = (self.belief.belief * self.obs_model[:, obs.value]).sum()
            
            # Hypothetical belief after observation
            temp_belief = BeliefState(self.belief.belief.copy())
            temp_belief.update(obs, self.obs_model)
            
            # Entropy after observation
            future_entropy = temp_belief.entropy()
            expected_future_entropy += prob_obs * future_entropy
            
            # Expected reward from eventual commitment
            confidence = temp_belief.confidence()
            commit_reward = (confidence * self.env.config.correct_reward + 
                           (1 - confidence) * self.env.config.incorrect_penalty)
            expected_future_reward += prob_obs * commit_reward
        
        # Information gain (entropy reduction)
        info_gain = current_entropy - expected_future_entropy
        
        # Total value = reward + weighted information gain
        total_value = expected_value + expected_future_reward + \
                     self.info_gain_weight * info_gain
        
        return total_value

### 3.4 Variational Free Energy ρ-POMDP Agent

Uses variational free energy as belief-state utility.

In [ ]:
class VFEAgent(BaseAgent):
    """ρ-POMDP agent using variational free energy.
    
    Variational Free Energy:
    F = E_q[Energy] - H(q)
      = -E_q[log P(o, s)] + E_q[log q(s)]
      = -E_q[log P(o|s)] - E_q[log P(s)] - H(q)
    
    For this simple environment:
    - P(s) = uniform prior
    - P(o|s) from observation model
    - q(s) is current belief
    
    Actions minimize expected free energy.
    """
    
    def __init__(self, env: MinimalInfoSeekingEnv, prior: Optional[np.ndarray] = None):
        """
        Args:
            env: Environment
            prior: Prior distribution over states P(s)
        """
        super().__init__(env)
        self.prior = prior if prior is not None else np.array([0.5, 0.5])
    
    def select_action(self) -> Action:
        """Select action that minimizes expected free energy."""
        commit_efe = self._expected_free_energy_commit()
        observe_efe = self._expected_free_energy_observe()
        
        # Choose action with lower expected free energy
        if observe_efe < commit_efe:
            return Action.OBSERVE
        else:
            return self.get_commit_action()
    
    def _expected_free_energy_commit(self) -> float:
        """Expected free energy of committing now.
        
        EFE for commit includes:
        - Negative expected reward (energy)
        - Current belief entropy is already known
        """
        # Energy term: negative expected reward
        expected_reward = self.expected_reward_of_commit()
        energy = -expected_reward
        
        # Epistemic term: KL[q(s)||p(s)]
        # For uniform prior, this is just entropy difference
        kl_term = np.sum(self.belief.belief * 
                        (np.log(self.belief.belief + 1e-10) - 
                         np.log(self.prior + 1e-10)))
        
        return energy + kl_term
    
    def _expected_free_energy_observe(self) -> float:
        """Expected free energy of observing, then committing optimally.
        
        EFE = E_q[E_p[Energy | o]] - E_q[H(q(s|o))]
        
        This combines:
        1. Pragmatic value (expected reward)
        2. Epistemic value (uncertainty reduction)
        """
        efe = self.env.config.observation_cost  # Immediate cost
        expected_entropy = 0.0
        expected_energy = 0.0
        
        for obs in [Observation.SIGNAL_A, Observation.SIGNAL_B]:
            # Probability of this observation
            prob_obs = (self.belief.belief * self.obs_model[:, obs.value]).sum()
            
            # Belief after observation
            temp_belief = BeliefState(self.belief.belief.copy())
            temp_belief.update(obs, self.obs_model)
            
            # Entropy of posterior (epistemic value)
            posterior_entropy = temp_belief.entropy()
            expected_entropy += prob_obs * posterior_entropy
            
            # Expected energy (negative reward)
            confidence = temp_belief.confidence()
            expected_reward = (confidence * self.env.config.correct_reward + 
                             (1 - confidence) * self.env.config.incorrect_penalty)
            expected_energy += prob_obs * (-expected_reward)
        
        # Expected Free Energy = Expected Energy - Expected Entropy
        # (Lower is better - we minimize EFE)
        efe += expected_energy - expected_entropy
        
        return efe

## 4. Evaluation Framework

In [ ]:
@dataclass
class EpisodeResult:
    """Results from a single episode."""
    agent_name: str
    num_observations: int
    final_belief_entropy: float
    final_confidence: float
    success: bool
    total_reward: float
    belief_history: List[np.ndarray]
    true_state: State
    committed_state: State

def run_episode(agent: BaseAgent, env: MinimalInfoSeekingEnv) -> EpisodeResult:
    """Run a single episode with the agent."""
    env.reset()
    agent.reset()
    
    while not env.done:
        action = agent.select_action()
        obs, reward, done = env.step(action)
        
        if not done:
            agent.update_belief(obs)
        else:
            # Episode ended with commitment
            committed_state = State.A if action == Action.COMMIT_A else State.B
            success = (committed_state == env.true_state)
            
            return EpisodeResult(
                agent_name=agent.__class__.__name__,
                num_observations=env.observation_count,
                final_belief_entropy=agent.belief.entropy(),
                final_confidence=agent.belief.confidence(),
                success=success,
                total_reward=env.total_reward,
                belief_history=agent.belief.history,
                true_state=env.true_state,
                committed_state=committed_state
            )

def run_experiment(agent_class, env: MinimalInfoSeekingEnv, 
                  num_episodes: int = 1000, **agent_kwargs) -> List[EpisodeResult]:
    """Run multiple episodes with an agent."""
    agent = agent_class(env, **agent_kwargs)
    results = []
    
    for _ in range(num_episodes):
        result = run_episode(agent, env)
        results.append(result)
    
    return results

def summarize_results(results: List[EpisodeResult]) -> Dict:
    """Compute summary statistics from episode results."""
    return {
        'agent': results[0].agent_name,
        'mean_observations': np.mean([r.num_observations for r in results]),
        'std_observations': np.std([r.num_observations for r in results]),
        'mean_final_entropy': np.mean([r.final_belief_entropy for r in results]),
        'mean_confidence': np.mean([r.final_confidence for r in results]),
        'success_rate': np.mean([r.success for r in results]),
        'mean_reward': np.mean([r.total_reward for r in results]),
        'std_reward': np.std([r.total_reward for r in results])
    }

## 5. Run Experiments

In [ ]:
# Create environment
config = EnvConfig(
    observation_accuracy=0.75,
    observation_cost=0.1,
    correct_reward=1.0,
    incorrect_penalty=-1.0
)
env = MinimalInfoSeekingEnv(config)

# Run experiments with all three agents
num_episodes = 1000

print("Running experiments...")
print(f"Episodes per agent: {num_episodes}")
print(f"Observation accuracy: {config.observation_accuracy}")
print(f"Observation cost: {config.observation_cost}\n")

myopic_results = run_experiment(MyopicAgent, env, num_episodes)
print("✓ Myopic Agent complete")

ig_results = run_experiment(InformationGainAgent, env, num_episodes, info_gain_weight=1.0)
print("✓ Information Gain Agent complete")

vfe_results = run_experiment(VFEAgent, env, num_episodes)
print("✓ VFE Agent complete\n")

# Summarize results
all_results = {
    'Myopic': summarize_results(myopic_results),
    'Information Gain': summarize_results(ig_results),
    'VFE': summarize_results(vfe_results)
}

# Display summary table
summary_df = pd.DataFrame(all_results).T
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(summary_df.to_string())
print("="*80)

## 6. Visualization and Analysis

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Epistemic Foraging Behavior Comparison', fontsize=16, fontweight='bold')

agent_results = {
    'Myopic': myopic_results,
    'Info Gain': ig_results,
    'VFE': vfe_results
}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# 1. Distribution of observation counts
ax = axes[0, 0]
for (name, results), color in zip(agent_results.items(), colors):
    obs_counts = [r.num_observations for r in results]
    ax.hist(obs_counts, alpha=0.6, label=name, bins=15, color=color)
ax.set_xlabel('Number of Observations')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Observation Counts')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Mean observations comparison
ax = axes[0, 1]
names = list(agent_results.keys())
means = [all_results[name]['mean_observations'] for name in names]
stds = [all_results[name]['std_observations'] for name in names]
ax.bar(names, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.set_ylabel('Mean Observations')
ax.set_title('Average Epistemic Foraging Behavior')
ax.grid(True, alpha=0.3, axis='y')

# 3. Success rates
ax = axes[0, 2]
success_rates = [all_results[name]['success_rate'] for name in names]
ax.bar(names, success_rates, color=colors, alpha=0.7)
ax.set_ylabel('Success Rate')
ax.set_title('Task Success Rate')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Chance')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 4. Final belief entropy
ax = axes[1, 0]
for (name, results), color in zip(agent_results.items(), colors):
    entropies = [r.final_belief_entropy for r in results]
    ax.hist(entropies, alpha=0.6, label=name, bins=15, color=color)
ax.set_xlabel('Final Belief Entropy (bits)')
ax.set_ylabel('Frequency')
ax.set_title('Uncertainty at Decision Time')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Mean total reward
ax = axes[1, 1]
rewards = [all_results[name]['mean_reward'] for name in names]
reward_stds = [all_results[name]['std_reward'] for name in names]
ax.bar(names, rewards, yerr=reward_stds, color=colors, alpha=0.7, capsize=5)
ax.set_ylabel('Mean Total Reward')
ax.set_title('Expected Utility Achieved')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.grid(True, alpha=0.3, axis='y')

# 6. Belief trajectory example
ax = axes[1, 2]
for (name, results), color in zip(agent_results.items(), colors):
    # Take first episode as example
    history = results[0].belief_history
    confidence_trajectory = [max(b) for b in history]
    ax.plot(confidence_trajectory, label=name, linewidth=2, color=color, marker='o')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Confidence (Max Belief)')
ax.set_title('Example Belief Convergence Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Statistical Analysis

In [ ]:
from scipy import stats

print("\n" + "="*80)
print("STATISTICAL COMPARISONS")
print("="*80 + "\n")

# Pairwise comparisons on number of observations
myopic_obs = [r.num_observations for r in myopic_results]
ig_obs = [r.num_observations for r in ig_results]
vfe_obs = [r.num_observations for r in vfe_results]

print("Number of Observations (t-tests):")
print("-" * 80)

# Myopic vs Info Gain
t_stat, p_val = stats.ttest_ind(myopic_obs, ig_obs)
print(f"Myopic vs Information Gain: t={t_stat:.3f}, p={p_val:.4f}")

# Myopic vs VFE
t_stat, p_val = stats.ttest_ind(myopic_obs, vfe_obs)
print(f"Myopic vs VFE: t={t_stat:.3f}, p={p_val:.4f}")

# Info Gain vs VFE
t_stat, p_val = stats.ttest_ind(ig_obs, vfe_obs)
print(f"Information Gain vs VFE: t={t_stat:.3f}, p={p_val:.4f}")

# Pairwise comparisons on total reward
myopic_reward = [r.total_reward for r in myopic_results]
ig_reward = [r.total_reward for r in ig_results]
vfe_reward = [r.total_reward for r in vfe_results]

print("\nTotal Reward (t-tests):")
print("-" * 80)

t_stat, p_val = stats.ttest_ind(myopic_reward, ig_reward)
print(f"Myopic vs Information Gain: t={t_stat:.3f}, p={p_val:.4f}")

t_stat, p_val = stats.ttest_ind(myopic_reward, vfe_reward)
print(f"Myopic vs VFE: t={t_stat:.3f}, p={p_val:.4f}")

t_stat, p_val = stats.ttest_ind(ig_reward, vfe_reward)
print(f"Information Gain vs VFE: t={t_stat:.3f}, p={p_val:.4f}")

print("\n" + "="*80)

## 8. Key Findings and Interpretation

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80 + "\n")

# Compare observation strategies
myopic_mean_obs = all_results['Myopic']['mean_observations']
ig_mean_obs = all_results['Information Gain']['mean_observations']
vfe_mean_obs = all_results['VFE']['mean_observations']

print("1. EPISTEMIC FORAGING BEHAVIOR")
print(f"   - Myopic agent observes {myopic_mean_obs:.2f} times on average")
print(f"   - Information Gain agent observes {ig_mean_obs:.2f} times on average")
print(f"   - VFE agent observes {vfe_mean_obs:.2f} times on average")

if ig_mean_obs > myopic_mean_obs:
    print(f"   → Info Gain explores {((ig_mean_obs/myopic_mean_obs - 1)*100):.1f}% more than Myopic")
if vfe_mean_obs > myopic_mean_obs:
    print(f"   → VFE explores {((vfe_mean_obs/myopic_mean_obs - 1)*100):.1f}% more than Myopic")
if abs(vfe_mean_obs - ig_mean_obs) > 0.1:
    direction = "more" if vfe_mean_obs > ig_mean_obs else "less"
    pct = abs((vfe_mean_obs/ig_mean_obs - 1)*100)
    print(f"   → VFE explores {pct:.1f}% {direction} than Information Gain")

# Compare rewards
print("\n2. EXPECTED UTILITY")
for name in ['Myopic', 'Information Gain', 'VFE']:
    mean_reward = all_results[name]['mean_reward']
    success_rate = all_results[name]['success_rate']
    print(f"   - {name}: {mean_reward:.3f} reward, {success_rate:.1%} success rate")

# Compare uncertainty
print("\n3. BELIEF CONVERGENCE")
for name in ['Myopic', 'Information Gain', 'VFE']:
    entropy = all_results[name]['mean_final_entropy']
    confidence = all_results[name]['mean_confidence']
    print(f"   - {name}: {entropy:.3f} bits entropy, {confidence:.1%} confidence at decision")

print("\n" + "="*80)

## 9. Next Steps

Based on these results:

1. **If agents show similar behavior**: The environment may be too simple. Consider:
   - Increasing observation cost
   - Decreasing observation accuracy
   - Adding more states

2. **If VFE differs from Info Gain**: You've found behavioral divergence. Investigate:
   - Under what parameter regimes do they differ most?
   - Which approach achieves better reward-information tradeoffs?
   - How does this scale to the Tiger problem?

3. **Ready to scale**: Once validated here, move to:
   - Tiger problem (3 actions, 2 states, richer structure)
   - Multi-step planning horizons
   - More complex observation models